[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/InterstellarNerd8760/lunar-atlas/blob/proxima/terrain-notebooks/notebooks/prepare-shackleton-site.ipynb)

# Shackleton rim (PGDA Site04): site, 5 m detail, slope, and horizon

The same LOLA 5 m/pixel Site04 map from the practice notebook, turned into app files in the `public/data/global-height` format. The map is in **south polar stereographic** coordinates: x/y in meters from the pole.

Outputs:
- `shackleton-site`: the whole 16 km site, averaged down to `SITE_PX` pixels.
- `shackleton-detail`: a small box at the full 5 m resolution.
- `shackleton-slope` (optional): slope in degrees on the same grid as the site.
- `shackleton-context`: a wide box from the 80 m polar map, for ridges on the horizon.

Runs on a free CPU runtime in Colab or Kaggle (on Kaggle, turn on **Internet**). **Runtime → Run all.**

In [ ]:
%pip -q install rasterio matplotlib requests

## 1. Settings

In [ ]:
SITE_PX = 1024              # whole 16 km site at 1024 px is about 15.6 m/px
DETAIL_KM = 4               # full-res 5 m box this many km wide (4 km = 800 x 800 px)
DETAIL_CENTER_LATLON = None # None = site center, or (lat, lon) like (-89.77, -171.9)
INCLUDE_SLOPE = True        # also export the PGDA slope map (degrees)
CONTEXT_KM = 200            # horizon box width; Shackleton crater itself is about 21 km across
CONTEXT_PX = 1024           # 200 km / 1024 px is about 195 m/px
ASSET_NAME = "shackleton"
USE_DRIVE = False           # True: save exports to Google Drive (Colab only)
DOWNLOAD_ZIP = True         # Colab: download the zip at the end

## 2. Setup
Helper functions. Nothing to change here.

In [ ]:
import json
import sys
import time
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

import matplotlib.pyplot as plt
import numpy as np
import rasterio
import rasterio.warp
import requests
from rasterio.enums import Resampling
from rasterio.windows import from_bounds

IN_COLAB = "google.colab" in sys.modules

# Raw downloads always stay on the runtime disk. Only finished exports go to Drive.
RAW = Path("data/raw")
OUT_ROOT = Path("data/work")
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_ROOT = Path("/content/drive/MyDrive/lunar-atlas")
EXPORT = OUT_ROOT / "exports" / ASSET_NAME
RAW.mkdir(parents=True, exist_ok=True)
EXPORT.mkdir(parents=True, exist_ok=True)

MOON_RADIUS_M = 1737400
# Moon lat/lon (same sphere as the global-height export).
MOON_GEOGRAPHIC = rasterio.crs.CRS.from_proj4("+proj=longlat +R=1737400 +no_defs")


def remote_size(url):
    """Ask the server how big a file is (HTTP HEAD) without downloading it."""
    response = requests.head(url, allow_redirects=True, timeout=30)
    response.raise_for_status()
    return int(response.headers.get("Content-Length", 0))


def download(url, name=None):
    """Download once into data/raw. Re-running the cell skips finished files."""
    path = RAW / (name or Path(url).name)
    if path.exists():
        return path
    temporary_path = path.with_suffix(path.suffix + ".part")
    with requests.get(url, stream=True, timeout=(15, 120)) as response:
        response.raise_for_status()
        with temporary_path.open("wb") as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                output.write(chunk)
    temporary_path.replace(path)
    return path


def save_grid(elevation, metadata, folder, name):
    """Write <name>.f32 (float32 little-endian, row-major) and <name>.json."""
    elevation = np.asarray(elevation, dtype="<f4")
    if not np.isfinite(elevation).any():
        raise ValueError(f"{name}: no valid elevations found.")
    folder.mkdir(parents=True, exist_ok=True)
    metadata = dict(metadata)
    metadata["minElevationM"] = float(np.nanmin(elevation))
    metadata["maxElevationM"] = float(np.nanmax(elevation))
    elevation.tofile(folder / f"{name}.f32")
    (folder / f"{name}.json").write_text(json.dumps(metadata, indent=2) + "\n")
    return metadata


def zip_folder(folder, zip_path):
    with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as bundle:
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                bundle.write(path, path.relative_to(folder.parent))
    return zip_path


def folder_size_mb(folder):
    return sum(p.stat().st_size for p in folder.rglob("*") if p.is_file()) / 1e6

print("Colab" if IN_COLAB else "Not Colab (Kaggle or local)", "| exports go to", EXPORT.resolve())


def polar_metadata(transform, width, height, crs, extra):
    """Metadata for a south-polar-stereographic grid, plus lat/lon facts."""
    left, top = transform * (0, 0)
    right, bottom = transform * (width, height)
    center_x, center_y = transform * (width / 2, height / 2)
    lon, lat = rasterio.warp.transform(crs, MOON_GEOGRAPHIC, [center_x], [center_y])
    # Sample the outline to find the lat/lon box the grid covers.
    edge = np.linspace(0, 1, 50)
    xs = np.concatenate([left + (right - left) * edge, np.full(50, right),
                         left + (right - left) * edge, np.full(50, left)])
    ys = np.concatenate([np.full(50, top), top + (bottom - top) * edge,
                         np.full(50, bottom), top + (bottom - top) * edge])
    lons, lats = rasterio.warp.transform(crs, MOON_GEOGRAPHIC, xs.tolist(), ys.tolist())
    pole_inside = left <= 0 <= right and bottom <= 0 <= top
    metadata = {
        **extra,
        "width": width, "height": height, "encoding": "float32-le",
        "units": "m", "order": "row-major", "nodataEncoding": "NaN",
        "referenceRadiusM": MOON_RADIUS_M,
        "globalExtentDegrees": None,
        "projection": "south polar stereographic (true scale at the pole), meters",
        "crsWkt": crs.to_wkt(),
        "affine": list(transform)[:6],
        "extentM": [left, bottom, right, top],
        "resolutionM": abs(transform.a),
        "centerLatLon": [round(lat[0], 6), round(((lon[0] + 180) % 360) - 180, 6)],
        "latRange": [-90.0 if pole_inside else round(min(lats), 6), round(max(lats), 6)],
        "lonRange": [-180.0, 180.0] if pole_inside else [round(min(lons), 6), round(max(lons), 6)],
    }
    return metadata


def read_grid(path, bounds, out_px, scale=1.0, offset=0.0):
    """Read a box (left, bottom, right, top in meters), averaged down to out_px wide."""
    with rasterio.open(path) as dataset:
        left, bottom, right, top = bounds
        # Keep the box inside the file.
        b = dataset.bounds
        left, bottom = max(left, b.left), max(bottom, b.bottom)
        right, top = min(right, b.right), min(top, b.top)
        window = from_bounds(left, bottom, right, top, dataset.transform).round_offsets().round_lengths()
        width = min(out_px, int(window.width))
        height = max(1, round(window.height * width / window.width))
        values = dataset.read(1, window=window, out_shape=(height, width), masked=True,
                              out_dtype="float32", resampling=Resampling.average)
        elevation = np.asarray((values * scale + offset).filled(np.nan), dtype="<f4")
        win_transform = dataset.window_transform(window)
        transform = win_transform * win_transform.scale(window.width / width, window.height / height)
        return elevation, transform, dataset.crs


def show(elevation, title):
    plt.figure(figsize=(5, 5))
    plt.imshow(elevation, cmap="terrain")
    plt.colorbar(label="Elevation (m)")
    plt.title(title)
    plt.show()

## 3. Download
About 112 MB in total. Sizes come from the server (HTTP HEAD).

In [ ]:
SITE_URL = "https://pgda.gsfc.nasa.gov/data/LOLA_5mpp/Site04/Site04_final_adj_5mpp_surf.tif"
SLOPE_URL = "https://pgda.gsfc.nasa.gov/data/LOLA_5mpp/Site04/Site04_final_adj_5mpp_slp.tif"
CONTEXT_URL = "https://imbrium.mit.edu/DATA/LOLA_GDR/POLAR/JP2/LDEM_80S_80M.JP2"
CONTEXT_LABEL = "https://imbrium.mit.edu/DATA/LOLA_GDR/POLAR/JP2/LDEM_80S_80M_JP2.LBL"
CONTEXT_SCALE = 0.5  # stored number x 0.5 = height in meters above the 1,737.4 km sphere
CITATION = ("Barker et al. (2021), Planetary and Space Science 203, 105119, "
            "doi:10.1016/j.pss.2020.105119")

for url in [SITE_URL] + ([SLOPE_URL] if INCLUDE_SLOPE else []) + [CONTEXT_URL]:
    print(f"{Path(url).name}  {remote_size(url) / 1e6:,.1f} MB")
site_path = download(SITE_URL)
slope_path = download(SLOPE_URL) if INCLUDE_SLOPE else None
context_path = download(CONTEXT_URL)
# The _AUX.XML file holds the scale. GDAL looks for "<file>.aux.xml".
download(CONTEXT_URL.replace(".JP2", "_AUX.XML"), name=context_path.name + ".aux.xml")

with rasterio.open(site_path) as dataset:
    site_box = tuple(dataset.bounds)
    site_crs = dataset.crs
    print("Site04:", dataset.width, "x", dataset.height, "px,", dataset.res[0], "m/px, box (m):", site_box)

## 4. Make the layers
`average` blends the source pixels inside each output pixel. The detail box keeps every 5 m pixel.

In [ ]:
def latlon_to_xy(lat, lon):
    xs, ys = rasterio.warp.transform(MOON_GEOGRAPHIC, site_crs, [lon], [lat])
    return xs[0], ys[0]


def box_around(x, y, km):
    half = km * 500
    return (x - half, y - half, x + half, y + half)


site_x = (site_box[0] + site_box[2]) / 2
site_y = (site_box[1] + site_box[3]) / 2
detail_x, detail_y = latlon_to_xy(*DETAIL_CENTER_LATLON) if DETAIL_CENTER_LATLON else (site_x, site_y)
common = {"productPage": "https://pgda.gsfc.nasa.gov/products/78", "siteId": "Site04",
          "siteName": "Shackleton rim", "resampling": "average"}
index = {}

# 1. The whole 16 km site, averaged down.
elevation, transform, crs = read_grid(site_path, site_box, SITE_PX)
meta = save_grid(elevation, polar_metadata(transform, elevation.shape[1], elevation.shape[0], crs, {
    "source": SITE_URL, **common, "role": "whole site", "sourceResolutionM": 5.0,
    "sourceScale": 1.0, "sourceOffset": 0.0, "citation": CITATION}), EXPORT, "shackleton-site")
index["site"] = meta
show(elevation, f"Whole site, {meta['resolutionM']:.1f} m/px")

# 2. Full 5 m detail around the chosen point.
detail, transform, crs = read_grid(site_path, box_around(detail_x, detail_y, DETAIL_KM), 10**6)
meta = save_grid(detail, polar_metadata(transform, detail.shape[1], detail.shape[0], crs, {
    "source": SITE_URL, **common, "role": "full-resolution detail", "sourceResolutionM": 5.0,
    "sourceScale": 1.0, "sourceOffset": 0.0, "citation": CITATION}), EXPORT, "shackleton-detail")
index["detail"] = meta
show(detail, f"Detail, {meta['resolutionM']:.0f} m/px")

# 3. Slope in degrees, same grid as the whole site.
if INCLUDE_SLOPE:
    slope, transform, crs = read_grid(slope_path, site_box, SITE_PX)
    meta = save_grid(slope, polar_metadata(transform, slope.shape[1], slope.shape[0], crs, {
        "source": SLOPE_URL, **common, "role": "slope (degrees, not meters)", "valueUnits": "deg",
        "sourceResolutionM": 5.0, "sourceScale": 1.0, "sourceOffset": 0.0, "citation": CITATION}),
        EXPORT, "shackleton-slope")
    meta["units"] = "deg"
    meta["minSlopeDeg"] = meta.pop("minElevationM")
    meta["maxSlopeDeg"] = meta.pop("maxElevationM")
    (EXPORT / "shackleton-slope.json").write_text(json.dumps(meta, indent=2) + "\n")
    index["slope"] = meta
    plt.imshow(slope, cmap="magma", vmin=0, vmax=30); plt.colorbar(label="Slope (°)"); plt.show()

# 4. Wide horizon box from the 80 m polar map.
context, transform, crs = read_grid(context_path, box_around(site_x, site_y, CONTEXT_KM), CONTEXT_PX,
                                    scale=CONTEXT_SCALE)
meta = save_grid(context, polar_metadata(transform, context.shape[1], context.shape[0], crs, {
    "source": CONTEXT_URL, "productLabel": CONTEXT_LABEL, "siteId": "Site04",
    "role": "horizon context", "sourceResolutionM": 80.0, "sourceScale": CONTEXT_SCALE,
    "sourceOffset": 0.0, "resampling": "average"}), EXPORT, "shackleton-context")
index["context"] = meta
show(context, f"Horizon context, {meta['resolutionM']:.0f} m/px")

summary = {name: {"file": f"shackleton-{name}.f32", "metadata": f"shackleton-{name}.json",
                  "width": m["width"], "height": m["height"], "resolutionM": m["resolutionM"],
                  "extentM": m["extentM"], "centerLatLon": m["centerLatLon"], "units": m["units"]}
           for name, m in index.items()}
(EXPORT / "index.json").write_text(json.dumps({"siteId": "Site04", "siteName": "Shackleton rim",
                                               "layers": summary}, indent=2) + "\n")
for name, m in summary.items():
    print(f"{name:8s} {m['width']}x{m['height']} at {m['resolutionM']:.1f} m/px, center {m['centerLatLon']}")

## 5. Zip and download
Unzip into `public/data/` to get `public/data/shackleton/`.

In [ ]:
zip_path = zip_folder(EXPORT, OUT_ROOT / f"{ASSET_NAME}.zip")
print(f"Exports: {folder_size_mb(EXPORT):.1f} MB in {EXPORT}")
print(f"Zip: {zip_path.stat().st_size / 1e6:.1f} MB at {zip_path}")
if DOWNLOAD_ZIP and IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
else:
    print("Download the zip from the file panel (Kaggle: Output tab).")